# Markovian Workload Generator

This notebook generates a synthetic workload directly from configurable
probability distributions. It does not read another workload to determine its
parameters.

The interarrival scale is calculated from `target_utilization`:

```text
mean interarrival =
    expected resources per job × mean runtime
    ─────────────────────────────────────────
          platform resources × target utilization
```

The generator writes two files:

- The generated workload JSON
- A separate JSON file containing the parameters used for that generation


In [52]:
#!/usr/bin/env python3
"""Generate a synthetic Markovian workload.

The arrival rate is controlled by target_utilization. The generator calculates
the exponential interarrival scale from the expected resource demand, expected
runtime, platform capacity, and utilization target:

    mean_interarrival =
        E[resources per job] * E[runtime]
        ----------------------------------
        nb_res * target_utilization

The generated job fields are:

- res: number of requested resources
- subtime: submission time in seconds
- reqtime: user-requested wall time in seconds
- runtime: actual execution time in seconds
- profile: key pointing to an entry in the top-level profiles dictionary
- user_id: synthetic user identifier
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np


@dataclass(frozen=True)
class GeneratorConfig:
    """Parameters for one generated workload."""

    num_jobs: int = 3000
    nb_res: int = 64
    target_utilization: float = 0.40

    mean_runtime: float = 7200.0
    resource_geometric_p: float = 0.35
    reqtime_bias: float = 1.20
    seed: int = 43

    # Every job stores this string in its "profile" field. It is a lookup key
    # into the top-level "profiles" dictionary, not a probability profile.
    profile_name: str = "100"

    # Retained for compatibility with the workload JSON schema.
    profile_cpu: int = 10_000_000_000_000_000_000_000
    profile_com: int = 0
    user_id: int = 0


def validate_config(config: GeneratorConfig) -> None:
    if config.num_jobs <= 0:
        raise ValueError("num_jobs must be positive")
    if config.nb_res <= 0:
        raise ValueError("nb_res must be positive")
    if not 0 < config.target_utilization <= 1:
        raise ValueError("target_utilization must be in (0, 1]")
    if config.mean_runtime <= 0:
        raise ValueError("mean_runtime must be positive")
    if not 0 < config.resource_geometric_p <= 1:
        raise ValueError("resource_geometric_p must be in (0, 1]")
    if config.reqtime_bias <= 0:
        raise ValueError("reqtime_bias must be positive")
    if not config.profile_name:
        raise ValueError("profile_name must not be empty")


def expected_resources_per_job(config: GeneratorConfig) -> float:
    """Return E[min(Geometric(p), nb_res)]."""
    p = config.resource_geometric_p
    q = 1.0 - p

    # For positive-support geometric X:
    # E[min(X, M)] = sum(k=1..M) P(X >= k)
    return (1.0 - q ** config.nb_res) / p


def mean_interarrival_from_utilization(
    config: GeneratorConfig,
) -> float:
    """Calculate the exponential interarrival scale."""
    return (
        expected_resources_per_job(config)
        * config.mean_runtime
        / (config.nb_res * config.target_utilization)
    )


def generate_workload(config: GeneratorConfig) -> dict[str, Any]:
    """Generate one workload using a reproducible NumPy random generator."""
    validate_config(config)

    rng = np.random.default_rng(config.seed)

    resources = rng.geometric(
        config.resource_geometric_p,
        size=config.num_jobs,
    )
    resources = np.minimum(resources, config.nb_res).astype(int)

    mean_interarrival = mean_interarrival_from_utilization(config)
    interarrivals = rng.exponential(
        scale=mean_interarrival,
        size=config.num_jobs,
    )
    subtimes = np.cumsum(interarrivals)

    runtimes = rng.exponential(
        scale=config.mean_runtime,
        size=config.num_jobs,
    )

    reqtimes = rng.exponential(
        scale=config.mean_runtime * config.reqtime_bias,
        size=config.num_jobs,
    )

    jobs = [
        {
            "job_id": index + 1,
            "res": int(resources[index]),
            "subtime": float(subtimes[index]),
            "reqtime": float(reqtimes[index]),
            "runtime": float(runtimes[index]),
            "profile": config.profile_name,
            "user_id": config.user_id,
        }
        for index in range(config.num_jobs)
    ]

    return {
        "nb_res": config.nb_res,
        "jobs": jobs,
        "profiles": {
            config.profile_name: {
                "cpu": config.profile_cpu,
                "com": config.profile_com,
                "type": "parallel_homogeneous",
            }
        },
    }


def write_workload(
    workload: dict[str, Any],
    output_path: str | Path,
) -> Path:
    """Write the generated workload as formatted JSON."""
    path = Path(output_path).expanduser()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(workload, indent=4),
        encoding="utf-8",
    )
    return path


def summarize(
    config: GeneratorConfig,
    workload: dict[str, Any],
) -> dict[str, float | int]:
    """Return generated and observed workload statistics."""
    jobs = workload["jobs"]

    resources = np.asarray(
        [job["res"] for job in jobs],
        dtype=float,
    )
    subtimes = np.asarray(
        [job["subtime"] for job in jobs],
        dtype=float,
    )
    runtimes = np.asarray(
        [job["runtime"] for job in jobs],
        dtype=float,
    )
    reqtimes = np.asarray(
        [job["reqtime"] for job in jobs],
        dtype=float,
    )

    interarrivals = np.diff(
        np.concatenate(([0.0], subtimes))
    )

    observed_utilization = (
        resources.mean()
        * runtimes.mean()
        / (config.nb_res * interarrivals.mean())
    )

    return {
        "num_jobs": config.num_jobs,
        "nb_res": config.nb_res,
        "target_utilization": config.target_utilization,
        "calculated_mean_interarrival": (
            mean_interarrival_from_utilization(config)
        ),
        "observed_mean_interarrival": float(
            interarrivals.mean()
        ),
        "observed_mean_resources": float(resources.mean()),
        "observed_mean_runtime": float(runtimes.mean()),
        "observed_mean_reqtime": float(reqtimes.mean()),
        "observed_offered_utilization": float(
            observed_utilization
        ),
    }


## Configuration

Change the values below before running the generation cell.

### What is `profile`?

Each job has a field such as:

```json
"profile": "100"
```

That value is simply a key pointing to the matching entry in the workload's
top-level `profiles` dictionary:

```json
"profiles": {
    "100": {
        "cpu": 10000000000000000000000,
        "com": 0,
        "type": "parallel_homogeneous"
    }
}
```

All jobs currently use the same profile. It is kept because the workload JSON
schema expects jobs to reference a profile. It is unrelated to the Markov
arrival model, utilization target, runtime distribution, or requested-time
bias.


In [53]:
CONFIG = GeneratorConfig(
    num_jobs=1,
    nb_res=64,
    target_utilization=0.40,
    mean_runtime=7200.0,
    resource_geometric_p=0.35,
    reqtime_bias=1.20,
    seed=6,
    profile_name="100",
)

OUTPUT_PATH = Path("workloads/dummy/SDSC-BLUE-2000-4.2-cln-0-3000.json")
PARAMETERS_PATH = OUTPUT_PATH.with_name(
    f"{OUTPUT_PATH.stem}-parameters.json"
)

print(
    "Calculated mean interarrival:",
    mean_interarrival_from_utilization(CONFIG),
    "seconds",
)


Calculated mean interarrival: 803.5714285705745 seconds


## Generate the workload

This cell writes the workload and the parameters used to generate it. The
parameter file is useful for reproducing the same workload later.


In [54]:
workload = generate_workload(CONFIG)
write_workload(workload, OUTPUT_PATH)

parameters = {
    "generator": "Markovian workload generator",
    "rng": "numpy.random.default_rng",
    "seed": CONFIG.seed,
    "draw_order": [
        "resources",
        "interarrivals",
        "runtimes",
        "requested_times",
    ],
    "num_jobs": CONFIG.num_jobs,
    "nb_res": CONFIG.nb_res,
    "target_utilization": CONFIG.target_utilization,
    "mean_runtime": CONFIG.mean_runtime,
    "resource_distribution": "geometric",
    "resource_geometric_p": (
        CONFIG.resource_geometric_p
    ),
    "expected_resources_per_job": (
        expected_resources_per_job(CONFIG)
    ),
    "interarrival_distribution": "exponential",
    "calculated_mean_interarrival": (
        mean_interarrival_from_utilization(CONFIG)
    ),
    "runtime_distribution": "exponential",
    "reqtime_distribution": "exponential",
    "reqtime_bias": CONFIG.reqtime_bias,
    "reqtime_mean": (
        CONFIG.mean_runtime * CONFIG.reqtime_bias
    ),
    "profile_name": CONFIG.profile_name,
    "profile_cpu": CONFIG.profile_cpu,
    "profile_com": CONFIG.profile_com,
    "user_id": CONFIG.user_id,
    "workload_output": str(OUTPUT_PATH),
}

PARAMETERS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)
PARAMETERS_PATH.write_text(
    json.dumps(parameters, indent=4),
    encoding="utf-8",
)

print(f"Wrote workload: {OUTPUT_PATH}")
print(f"Wrote parameters: {PARAMETERS_PATH}")
print()
print(json.dumps(parameters, indent=4))
print()
summarize(CONFIG, workload)


Wrote workload: workloads/dummy/SDSC-BLUE-2000-4.2-cln-0-3000.json
Wrote parameters: workloads/dummy/SDSC-BLUE-2000-4.2-cln-0-3000-parameters.json

{
    "generator": "Markovian workload generator",
    "rng": "numpy.random.default_rng",
    "seed": 6,
    "draw_order": [
        "resources",
        "interarrivals",
        "runtimes",
        "requested_times"
    ],
    "num_jobs": 1,
    "nb_res": 64,
    "target_utilization": 0.4,
    "mean_runtime": 7200.0,
    "resource_distribution": "geometric",
    "resource_geometric_p": 0.35,
    "expected_resources_per_job": 2.8571428571398205,
    "interarrival_distribution": "exponential",
    "calculated_mean_interarrival": 803.5714285705745,
    "runtime_distribution": "exponential",
    "reqtime_distribution": "exponential",
    "reqtime_bias": 1.2,
    "reqtime_mean": 8640.0,
    "profile_name": "100",
    "profile_cpu": 10000000000000000000000,
    "profile_com": 0,
    "user_id": 0,
    "workload_output": "workloads/dummy/SDSC-BLUE

{'num_jobs': 1,
 'nb_res': 64,
 'target_utilization': 0.4,
 'calculated_mean_interarrival': 803.5714285705745,
 'observed_mean_interarrival': 147.41525164491037,
 'observed_mean_resources': 2.0,
 'observed_mean_runtime': 4339.537409929727,
 'observed_mean_reqtime': 4630.034605927254,
 'observed_offered_utilization': 0.919922074189167}